# NSE — Extras (Scheme B · seeds/CIs · backtest)

Deferred robustness experiments, split out of `nse_matrix.ipynb` (which holds the primary Scheme-A
transfer result). Self-contained — same setup as the main NSE notebook. Reads/extends the same
`nse_matrix.csv` on S3, so it resumes alongside the Scheme-A runs.

Sections: **Scheme B** (spread-relative labels, E4) · **multi-seed + bootstrap CIs** · **cost-aware backtest** (E8).
Needs GPU + secrets `GH_PAT`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` (attach them on Kaggle).


## 1. Runtime check

In [ ]:
import torch, platform
print("Python:", platform.python_version(), "| Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get the project code (GH_PAT secret)

In [ ]:
import sys, subprocess, pathlib
REPO_URL = "https://github.com/rajjoseph48/nse-lob-capstone.git"
REPO_DIR = "nse-lob-capstone"
def _get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    import os
    return os.environ.get(name, "")

GITHUB_TOKEN = _get_secret("GH_PAT")
def add_modeling(base):
    base = pathlib.Path(base)
    for cand in (base / "modeling", base):
        if (cand / "models.py").exists():
            sys.path.insert(0, str(cand.resolve()))
            return str(cand.resolve())
    return None
path = None
for c in (".", REPO_DIR, "/kaggle/working/" + REPO_DIR, "/content/" + REPO_DIR):
    path = add_modeling(c)
    if path:
        break
if not path:
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url], check=True)
    path = add_modeling(REPO_DIR)
print("modeling/ on sys.path:", path)

## 3. Install deps (Mamba kernel + boto3)

In [ ]:
import subprocess, sys
def pipq(*pkgs): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pipq("boto3")
pipq("ninja", "packaging", "setuptools", "wheel")
pipq("--no-build-isolation", "causal-conv1d")
pipq("--no-build-isolation", "mamba-ssm")
try:
    import mamba_ssm
    print("install step done | mamba-ssm", mamba_ssm.__version__)
except Exception as e:
    print("install step done | mamba-ssm NOT importable -> pure-PyTorch fallback:", repr(e))

## 4. Download NSE Dhan data from S3 (AWS_* secrets)

In [ ]:
import os, boto3, pathlib
def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, "")

os.environ["AWS_ACCESS_KEY_ID"] = _secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = _secret("AWS_SECRET_ACCESS_KEY")
BUCKET, PREFIX, REGION = "lob-capstone-data", "lob-data/dhan/", "ap-south-2"
DATA_DIR_NSE = "nse_data/dhan"
pathlib.Path(DATA_DIR_NSE).mkdir(parents=True, exist_ok=True)
s3 = boto3.client("s3", region_name=REGION)
n = 0
for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX).get("Contents", []):
    if o["Size"] == 0:
        continue
    dst = os.path.join(DATA_DIR_NSE, o["Key"].split("/")[-1])
    if not os.path.exists(dst):
        s3.download_file(BUCKET, o["Key"], dst)
    n += 1
print(f"{n} parquet files in {DATA_DIR_NSE}")

## 5. Metrics + naive baselines

In [ ]:
import numpy as np, torch
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             matthews_corrcoef, confusion_matrix)
from fi2010_dataset import make_loader
from train import DEVICE

CLASS_NAMES = ["Down", "Stat", "Up"]

def collect_preds(model, ds, batch_size=256):
    model = model.to(DEVICE).eval()
    loader = make_loader(ds, batch_size=batch_size, shuffle=False)
    preds, labels = [], []
    with torch.no_grad():
        for X, y in loader:
            preds.append(model(X.to(DEVICE)).argmax(1).cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(preds), np.concatenate(labels)

def full_metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1, 2], zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
            "mcc": float(matthews_corrcoef(y_true, y_pred)),
            "per_class": {CLASS_NAMES[i]: {"precision": float(p[i]), "recall": float(r[i]),
                                           "f1": float(f[i])} for i in range(3)},
            "confusion": confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist()}

def naive_baselines(y_train, y_test):
    maj = int(np.bincount(y_train, minlength=3).argmax())
    wf1 = lambda yp: f1_score(y_test, yp, average="weighted", zero_division=0)
    rng = np.random.default_rng(0)
    probs = np.bincount(y_train, minlength=3) / len(y_train)
    return {"baseline_majority_wf1": round(wf1(np.full_like(y_test, maj)), 4),
            "baseline_stat_wf1": round(wf1(np.full_like(y_test, 1)), 4),
            "baseline_random_wf1": round(wf1(rng.choice(3, size=len(y_test), p=probs)), 4)}

## 6. Runner + config (resumes from the same nse_matrix.csv on S3)

In [ ]:
import time, json, pathlib, pandas as pd
from nse_dataset import load_nse
from models import build_model
from train import train, save_checkpoint
from nbenv import s3_client, s3_put, s3_pull

RESULTS_DIR = pathlib.Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
CKPT_DIR = pathlib.Path("checkpoints/nse"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / "nse_matrix.csv"
MODEL_LR = {"deeplob": 1e-3, "mlplob": 1e-3, "tlob": 1e-4, "mambalob": 3e-4}
MODELS = ["deeplob", "mlplob", "tlob", "mambalob"]
SYMBOLS = ["NIFTY", "BANKNIFTY"]
EPOCHS = 20
S3_BUCKET, S3_PREFIX = "lob-capstone-data", "reproduction/nse"
_s3 = s3_client()
s3_pull(_s3, RESULTS_CSV, S3_BUCKET, S3_PREFIX)   # bring in Scheme-A progress to skip/dedup

def run_nse(model_name, symbol, horizon, label_scheme="A", seed=0,
            epochs=20, patience=3, batch_size=128, seq_len=100, save_to_csv=True):
    from stats import set_seed
    set_seed(seed)
    sch = "" if label_scheme.upper() == "A" else f"_{label_scheme.upper()}"
    tag = f"{model_name}_{symbol}_h{horizon}{sch}_s{seed}"
    lr = MODEL_LR.get(model_name, 1e-3)
    print("=" * 72); print(" ", tag, f"(lr={lr}, scheme={label_scheme})"); print("=" * 72)
    tr, vl, te = load_nse(symbol=symbol, horizon=horizon, seq_len=seq_len,
                          alpha=1e-5, label_scheme=label_scheme, data_dir=DATA_DIR_NSE)
    model = build_model(model_name, seq_len=seq_len, n_features=40)
    n_params = sum(p.numel() for p in model.parameters())
    t0 = time.time()
    hist = train(model, tr, vl, epochs=epochs, patience=patience, batch_size=batch_size, lr=lr, verbose=True)
    elapsed = time.time() - t0
    y_pred, y_true = collect_preds(model, te)
    mt = full_metrics(y_true, y_pred)
    bl = naive_baselines(tr.y.numpy(), y_true)
    row = {"model": model_name, "symbol": symbol, "horizon": horizon,
           "label_scheme": label_scheme.upper(), "seed": seed, "n_params": n_params,
           "best_epoch": hist["best_epoch"], "epochs_run": len(hist["val_f1"]),
           "best_val_f1": round(max(hist["val_f1"]), 4),
           "test_accuracy": round(mt["accuracy"], 4), "test_macro_f1": round(mt["macro_f1"], 4),
           "test_weighted_f1": round(mt["weighted_f1"], 4), "test_mcc": round(mt["mcc"], 4),
           **bl, "train_time_s": round(elapsed, 1)}
    if save_to_csv:
        ckpt = CKPT_DIR / f"{tag}.pt"; save_checkpoint(model, str(ckpt)); row["checkpoint"] = str(ckpt)
        df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        df.to_csv(RESULTS_CSV, index=False)
        for _p in (RESULTS_CSV, ckpt):
            s3_put(_s3, _p, S3_BUCKET, S3_PREFIX)
    print(f"  -> macro_f1={mt['macro_f1']:.4f} weighted_f1={mt['weighted_f1']:.4f} acc={mt['accuracy']:.4f} ({elapsed:.0f}s)")
    return row, mt, (y_true, y_pred)

def _done_keys():
    if not RESULTS_CSV.exists():
        return set()
    d = pd.read_csv(RESULTS_CSV)
    for col, default in [("label_scheme", "A"), ("seed", 0)]:
        if col not in d.columns:
            d[col] = default
    return {(r.model, r.symbol, int(r.horizon), r.label_scheme, int(r.seed)) for r in d.itertuples()}

## 7. Scheme B — spread-relative labels (E4): the finding is the *label balance*

θ = mean(spread/mid) over train (≈ transaction cost). On NSE most moves never clear the spread, so
Scheme B is Stationary-dominated — that imbalance *is* the economic result (no cost-clearing signal),
tying to the backtest (net ≈ −cost). Report the balance table (no training), then a small h200-only
confirmatory train.

In [ ]:
import numpy as np
rows = []
for sym in SYMBOLS:
    for h in [50, 100, 200]:
        tr, _, _ = load_nse(symbol=sym, horizon=h, seq_len=100, label_scheme="B", data_dir=DATA_DIR_NSE)
        c = np.bincount(tr.y.numpy(), minlength=3); p = c / c.sum() * 100
        rows.append({"symbol": sym, "horizon": h, "down_%": round(p[0], 1),
                     "stat_%": round(p[1], 1), "up_%": round(p[2], 1), "n_train": int(c.sum())})
bal = pd.DataFrame(rows); bal.to_csv(RESULTS_DIR / "scheme_b_balance.csv", index=False)
s3_put(_s3, RESULTS_DIR / "scheme_b_balance.csv", S3_BUCKET, S3_PREFIX)
print(bal.to_string(index=False))
print("\nScheme B is Stationary-dominated -> most NSE moves are within the spread (no cost-clearing signal).")

**Confirmatory train (Scheme B, h200 only).** ~3 runs on NIFTY; widen `SYMS_B`/`MODELS_B` if desired.

In [ ]:
MODELS_B = ["deeplob", "tlob", "mambalob"]
SYMS_B = ["NIFTY"]
done = _done_keys()
for mdl in MODELS_B:
    for sym in SYMS_B:
        if (mdl, sym, 200, "B", 0) in done:
            print(f"skip {mdl} {sym} h200 B (in CSV)"); continue
        run_nse(mdl, sym, 200, label_scheme="B", seed=0, epochs=EPOCHS)
pd.read_csv(RESULTS_CSV).query("label_scheme == 'B'")

## 8. Multi-seed + bootstrap CIs (NIFTY, Scheme A)
Each headline config over 3 seeds → macro-F1 mean ± std + bootstrap 95% CI (seed 0's test preds).

In [ ]:
from stats import seed_summary, bootstrap_ci
SEEDS = [0, 1, 2]
HEADLINE_SYMBOL, HEADLINE_HORIZON = "NIFTY", 10
seed_rows = []
for mdl in MODELS:
    macro_by_seed, first_preds = [], None
    for s in SEEDS:
        _, mt, (yt, yp) = run_nse(mdl, HEADLINE_SYMBOL, HEADLINE_HORIZON,
                                  label_scheme="A", seed=s, epochs=EPOCHS, save_to_csv=False)
        macro_by_seed.append(mt["macro_f1"])
        if first_preds is None:
            first_preds = (yt, yp)
    summ = seed_summary(macro_by_seed); ci = bootstrap_ci(*first_preds, average="macro", n_boot=1000)
    seed_rows.append({"model": mdl, "symbol": HEADLINE_SYMBOL, "horizon": HEADLINE_HORIZON,
                      "macro_f1_mean": round(summ["mean"], 4), "macro_f1_std": round(summ["std"], 4),
                      "seeds": summ["values"], "ci_low": round(ci["ci_low"], 4), "ci_high": round(ci["ci_high"], 4)})
    print(f"{mdl}: macro-F1 {summ['mean']:.4f} ± {summ['std']:.4f}  95% CI [{ci['ci_low']:.4f}, {ci['ci_high']:.4f}]")
sdf = pd.DataFrame(seed_rows); sdf.to_csv(RESULTS_DIR / "nse_seeds.csv", index=False)
s3_put(_s3, RESULTS_DIR / "nse_seeds.csv", S3_BUCKET, S3_PREFIX)
sdf

## 9. Cost-aware backtest (E8) — signal quality, not a strategy
P(up)/P(down) > τ → long/short, hold the horizon, ~5 bps round-trip cost; sweep τ. Per-trade IR
(mean/std), hit-rate, net bps. Picks the highest-lift Scheme-A config and reloads its checkpoint.

In [ ]:
import os, torch, matplotlib.pyplot as plt
from backtest import run_backtest
res = pd.read_csv(RESULTS_CSV)
resA = res[res.get("label_scheme", "A") == "A"].copy()
resA["best_bl"] = resA[["baseline_majority_wf1", "baseline_stat_wf1", "baseline_random_wf1"]].max(axis=1)
resA["lift"] = resA["test_weighted_f1"] - resA["best_bl"]
best = resA.sort_values("lift").iloc[-1]
mdl, sym, h = best.model, best.symbol, int(best.horizon)
print(f"Backtesting highest-lift config: {mdl} {sym} h{h} (lift={best.lift:.4f})")
tr, vl, te = load_nse(symbol=sym, horizon=h, seq_len=100, data_dir=DATA_DIR_NSE)
model = build_model(mdl, seq_len=100, n_features=40)
ckpt = f"checkpoints/nse/{mdl}_{sym}_h{h}_s0.pt"
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); print("loaded", ckpt)
else:
    print("checkpoint missing — retraining"); train(model, tr, vl, epochs=20, patience=3,
        batch_size=128, lr=MODEL_LR.get(mdl, 1e-3), verbose=False)
bt = run_backtest(model, te, horizon=h, taus=(0.4, 0.5, 0.6, 0.7, 0.8), cost_bps=5.0)
print(bt.to_string(index=False))
bt.to_csv(RESULTS_DIR / f"backtest_{mdl}_{sym}_h{h}.csv", index=False)
s3_put(_s3, RESULTS_DIR / f"backtest_{mdl}_{sym}_h{h}.csv", S3_BUCKET, S3_PREFIX)
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot(bt.tau, bt.hit_rate, "o-"); ax[0].axhline(0.5, ls="--", c="grey"); ax[0].set(title="Hit rate vs τ", xlabel="τ", ylabel="hit rate")
ax[1].plot(bt.tau, bt.net_bps_mean, "s-"); ax[1].axhline(0, ls="--", c="grey"); ax[1].set(title="Mean net return vs τ", xlabel="τ", ylabel="bps / trade")
ax[2].plot(bt.tau, bt.trades, "^-"); ax[2].set(title="Trades vs τ", xlabel="τ", ylabel="# trades")
for a in ax: a.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR / f"fig_backtest_{mdl}_{sym}_h{h}.png", dpi=150, bbox_inches="tight")
s3_put(_s3, RESULTS_DIR / f"fig_backtest_{mdl}_{sym}_h{h}.png", S3_BUCKET, S3_PREFIX); plt.show()